# 📘 Session 5: Modules, Packages, `pip`, and Virtual Environments
### Duration: ~1.5 Hours

---

**Topics Covered:**
1. Modules — Creating and importing your own code
2. Packages — Organizing modules into hierarchies
3. `pip` — Installing third-party packages (NumPy, Pandas, Scikit-Learn)
4. Virtual Environments — Isolating project dependencies
5. Best Practices

---

**Why This Session Matters for Data Science:**
- **Modules** — Organize your code into reusable pieces (data loaders, preprocessors, models)
- **`pip`** — Install the entire data science ecosystem (NumPy, Pandas, Scikit-Learn, TensorFlow)
- **Virtual Environments** — Each project gets its own isolated dependencies; no version conflicts
- **Packages** — Structure large projects professionally

> Every data scientist works with external libraries daily. Understanding how to manage them is essential for moving from notebook scripts to production code.

---
## 1. Modules — Writing Reusable Code

A **module** is a Python file (`.py`) containing functions, classes, or variables that you can **import and reuse** elsewhere.

### Why Use Modules?

| Problem | Solution |
|---------|----------|
| Repeating same code in multiple notebooks | Write once in a module, import everywhere |
| Giant 500-line notebook | Split into logical modules |
| Hard to test code | Modules are easier to test |
| Team collaboration | Modules make code shareable |
| Switching between projects | Reuse modules across projects |

### Module Basics

**File: `math_utils.py`**
```python
"""Math utilities for common calculations."""

def add(a, b):
    """Add two numbers."""
    return a + b

def multiply(a, b):
    return a * b

PI = 3.14159
```

**File: `main.py`** (or notebook cell)
```python
import math_utils

result = math_utils.add(5, 3)      # 8
area = math_utils.PI * r ** 2      # Use constant
```

### Import Styles

| Style | Example | Use When |
|-------|---------|----------|
| `import module` | `import math_utils` then `math_utils.add()` | Default, clear what comes from where |
| `import module as alias` | `import pandas as pd` then `pd.read_csv()` | Long names (convention: `np`, `pd`) |
| `from module import name` | `from math_utils import add` then `add()` | Using one or two functions repeatedly |
| `from module import *` | `from math_utils import *` then `add()` | **❌ AVOID** — unclear what's imported |

> ⚠️ **Special Case — `from module import *` is dangerous**: It imports EVERYTHING, pollutes namespace, and makes code unclear. Always specify what you need: `from module import add, multiply`.

> ⚠️ **Special Case — Circular imports**: If `module_a` imports `module_b` and vice versa, Python raises `ImportError`. Redesign to avoid circular dependencies.

> **Data Science relevance**: All NumPy/Pandas/Scikit-Learn code uses `import numpy as np`, `import pandas as pd`. Following this convention makes your code instantly readable to other data scientists.

### The `__name__` Variable

Every Python file has a special variable `__name__`:
- `__name__ == '__main__'` when the file is **run directly**
- `__name__ == 'module_name'` when the file is **imported**

**Usage: Run code ONLY when file is executed directly**

```python
# File: data_loader.py

def load_data(filename):
    """Load CSV data."""
    import pandas as pd
    return pd.read_csv(filename)

# Test code — runs only when executed directly, not when imported!
if __name__ == '__main__':
    data = load_data('sample.csv')
    print(f"Loaded {len(data)} rows")
```

When you `import data_loader`, the `if __name__ == '__main__':` block is **skipped**. This prevents test code from running during import.

> ⚠️ **Special Case — Always use this pattern for modules**: Without it, importing a module will execute all its code (including test/demo code). This can cause side effects, slow imports, or errors.

In [2]:
# --- Create and use a simple module ---

# First, create the module file
module_code = '''"""Simple math utilities."""

def add(a, b):
    """Add two numbers."""
    return a + b

def multiply(a, b):
    """Multiply two numbers."""
    return a * b

def factorial(n):
    """Calculate factorial of n."""
    if n <= 1:
        return 1
    return n * factorial(n - 1)

PI = 3.14159
E = 2.71828

if __name__ == '__main__':
    print("Module is being run directly")
    print(f"5! = {factorial(5)}")
'''

with open('math_utils.py', 'w') as f:
    f.write(module_code)

print("✅ Created math_utils.py")

# Now import and use it
import math_utils

print(f"\n5 + 3 = {math_utils.add(5, 3)}")
print(f"5 * 3 = {math_utils.multiply(5, 3)}")
print(f"5! = {math_utils.factorial(5)}")
print(f"PI ≈ {math_utils.PI}")

# Check __name__
print(f"\nmath_utils.__name__ = {math_utils.__name__}")
print(f"__name__ here (notebook) = {__name__}")

✅ Created math_utils.py

5 + 3 = 8
5 * 3 = 15
5! = 120
PI ≈ 3.14159

math_utils.__name__ = math_utils
__name__ here (notebook) = __main__


In [3]:
# --- Import Styles Demonstration ---

# Style 1: import module
import math_utils as mu
result1 = mu.add(10, 5)
print(f"Style 1 (alias): 10 + 5 = {result1}")

# Style 2: from module import specific names
from math_utils import factorial, PI
result2 = factorial(4)
print(f"Style 2 (specific): 4! = {result2}")
print(f"Using imported PI: {PI}")

# Style 3: alias on import
from math_utils import multiply as mul
result3 = mul(7, 6)
print(f"Style 3 (aliased function): 7 * 6 = {result3}")

# Don't do this (poor practice):
# from math_utils import *   # Now what's available? Unclear!

Style 1 (alias): 10 + 5 = 15
Style 2 (specific): 4! = 24
Using imported PI: 3.14159
Style 3 (aliased function): 7 * 6 = 42


### Docstrings and Module Documentation

**Module docstring** (at the top of the file) explains what the module does:

```python
"""
Data preprocessing utilities for machine learning.

This module provides functions for:
- Handling missing values
- Encoding categorical variables
- Scaling features

Example:
    from preprocessing import impute_missing
    
    clean_data = impute_missing(raw_data, strategy='mean')
"""
```

**Function docstrings** explain what each function does (see Session 2 for details).

**Access documentation with `help()`**:
```python
import math_utils
help(math_utils)          # Show module docstring
help(math_utils.add)      # Show function docstring
```

> **Data Science relevance**: Good documentation is critical for team projects. Future-you (6 months later) will be grateful for clear docs on what each module does.

In [4]:
# --- Module with Documentation ---

preprocessing_code = '''"""Data preprocessing utilities for machine learning.

This module provides common data cleaning operations used in ML pipelines.

Functions:
    normalize(data): Scale data to 0-1 range
    remove_outliers(data): Remove values beyond 3 sigma
    
Example:
    from preprocessing import normalize
    cleaned = normalize(raw_data)
"""

def normalize(data):
    """Normalize data to 0-1 range using min-max scaling.
    
    Args:
        data: List or array of numbers
    
    Returns:
        List of normalized values
    
    Example:
        >>> normalize([1, 2, 3, 4, 5])
        [0.0, 0.25, 0.5, 0.75, 1.0]
    """
    min_val = min(data)
    max_val = max(data)
    return [(x - min_val) / (max_val - min_val) for x in data]

def remove_outliers(data):
    """Remove values beyond 3 standard deviations from mean.
    
    Args:
        data: List of numbers
    
    Returns:
        List without outliers
    """
    import statistics
    mean = statistics.mean(data)
    std = statistics.stdev(data)
    return [x for x in data if abs(x - mean) <= 3 * std]

if __name__ == '__main__':
    test_data = [1, 2, 3, 4, 5, 100]  # 100 is outlier
    print(f"Original: {test_data}")
    print(f"Normalized: {normalize(test_data)}")
    print(f"Without outliers: {remove_outliers(test_data)}")
'''

with open('preprocessing.py', 'w') as f:
    f.write(preprocessing_code)

print("✅ Created preprocessing.py")

# Use it
from preprocessing import normalize, remove_outliers

test_data = [10, 20, 30, 40, 50]
print(f"\nOriginal: {test_data}")
print(f"Normalized: {normalize(test_data)}")

# View help
print("\n--- Help on normalize function ---")
help(normalize)

✅ Created preprocessing.py

Original: [10, 20, 30, 40, 50]
Normalized: [0.0, 0.25, 0.5, 0.75, 1.0]

--- Help on normalize function ---
Help on function normalize in module preprocessing:

normalize(data)
    Normalize data to 0-1 range using min-max scaling.

    Args:
        data: List or array of numbers

    Returns:
        List of normalized values

    Example:
        >>> normalize([1, 2, 3, 4, 5])
        [0.0, 0.25, 0.5, 0.75, 1.0]



---
## 2. Packages — Organizing Modules

A **package** is a directory containing Python modules and a special `__init__.py` file.

### Package Structure

```
my_project/
├── data_science/          ← Package (directory)
│   ├── __init__.py        ← Makes it a package
│   ├── preprocessing.py   ← Module
│   ├── models.py          ← Module
│   └── visualization.py   ← Module
├── notebooks/
│   └── analysis.ipynb
└── requirements.txt       ← Dependencies
```

### The `__init__.py` File

Makes a directory a **package** so Python recognizes it.

**Empty `__init__.py`** (minimal):
```python
# Marks this directory as a package
```

**With exports** (common):
```python
"""Data science package."""

from .preprocessing import normalize, remove_outliers
from .models import train_model, evaluate

__version__ = '1.0.0'
__all__ = ['normalize', 'remove_outliers', 'train_model', 'evaluate']
```

### Importing from Packages

| Import | Accesses |
|--------|----------|
| `import data_science.preprocessing` | `data_science.preprocessing.normalize()` |
| `from data_science import preprocessing` | `preprocessing.normalize()` |
| `from data_science.preprocessing import normalize` | `normalize()` |
| `from data_science import normalize` | `normalize()` (if in `__init__.py`) |

> ⚠️ **Special Case — Relative imports with `.`**: Inside a package, use relative imports with dot notation.
> - `from .preprocessing import normalize` — import from sibling module
> - `from ..utils import helper` — import from parent package
> 
> This only works when importing; can't run a module directly with relative imports.

> **Data Science relevance**: Real projects (TensorFlow, Scikit-Learn) are organized as packages. Understanding this structure helps you navigate their source code.

In [5]:
# --- Create a Package ---
import os
from pathlib import Path

# Create package directory
pkg_dir = Path('ml_tools')
pkg_dir.mkdir(exist_ok=True)

# Create __init__.py
init_code = '''"""Machine learning tools package.

A collection of utilities for data preprocessing and model evaluation.

Modules:
    preprocessing: Data cleaning and transformation
    metrics: Model evaluation metrics
"""

__version__ = '1.0.0'
'''

with open(pkg_dir / '__init__.py', 'w') as f:
    f.write(init_code)

# Create preprocessing module
preprocess_code = '''"""Data preprocessing utilities."""

def clean_data(data, remove_nulls=True):
    """Remove null values from data."""
    if remove_nulls:
        return [x for x in data if x is not None]
    return data

def scale_data(data):
    """Scale data to 0-1 range."""
    min_val = min(data)
    max_val = max(data)
    return [(x - min_val) / (max_val - min_val) for x in data]
'''

with open(pkg_dir / 'preprocessing.py', 'w') as f:
    f.write(preprocess_code)

# Create metrics module
metrics_code = '''"""Model evaluation metrics."""

def accuracy(y_true, y_pred):
    """Calculate accuracy: fraction of correct predictions."""
    correct = sum(1 for true, pred in zip(y_true, y_pred) if true == pred)
    return correct / len(y_true)

def precision(y_true, y_pred):
    """Calculate precision: true positives / (true + false positives)."""
    tp = sum(1 for true, pred in zip(y_true, y_pred) if true == 1 and pred == 1)
    fp = sum(1 for true, pred in zip(y_true, y_pred) if true == 0 and pred == 1)
    return tp / (tp + fp) if (tp + fp) > 0 else 0
'''

with open(pkg_dir / 'metrics.py', 'w') as f:
    f.write(metrics_code)

print("✅ Created package ml_tools with:")
for file in sorted(pkg_dir.iterdir()):
    print(f"   - {file.name}")

✅ Created package ml_tools with:
   - __init__.py
   - metrics.py
   - preprocessing.py


In [6]:
# --- Import from Package ---

# Style 1: Full path
import ml_tools.preprocessing
data = [10, 20, 30, 40]
scaled = ml_tools.preprocessing.scale_data(data)
print(f"Style 1: {data} → {scaled}")

# Style 2: Import module
from ml_tools import metrics
y_true = [1, 0, 1, 1, 0]
y_pred = [1, 0, 1, 0, 0]
acc = metrics.accuracy(y_true, y_pred)
print(f"\nStyle 2: Accuracy = {acc:.2%}")

# Style 3: Import function directly
from ml_tools.preprocessing import clean_data
messy = [1, None, 2, None, 3]
clean = clean_data(messy)
print(f"\nStyle 3: {messy} → {clean}")

# Explore package
print(f"\nPackage location: {ml_tools.__file__}")
print(f"Package version: {ml_tools.__version__}")

Style 1: [10, 20, 30, 40] → [0.0, 0.3333333333333333, 0.6666666666666666, 1.0]

Style 2: Accuracy = 80.00%

Style 3: [1, None, 2, None, 3] → [1, 2, 3]

Package location: c:\python_AI_ML\ml_tools\__init__.py
Package version: 1.0.0


---
## 3. `pip` — Installing Third-Party Packages

**`pip`** (Pip Installs Packages) is Python's package manager. It downloads and installs packages from **PyPI** (Python Package Index).

### Essential Commands

| Command | Description |
|---------|-------------|
| `pip install package` | Install latest version |
| `pip install package==1.2.3` | Install specific version |
| `pip install package>=1.0` | Install version 1.0 or higher |
| `pip install -r requirements.txt` | Install all packages from file |
| `pip list` | List installed packages |
| `pip show package` | Show package details |
| `pip search term` | Search PyPI (deprecated, use website) |
| `pip uninstall package` | Remove package |
| `pip install --upgrade package` | Update to latest version |

### Common Data Science Packages

| Package | Purpose | Install |
|---------|---------|----------|
| NumPy | Numerical computing | `pip install numpy` |
| Pandas | Data analysis | `pip install pandas` |
| Matplotlib | Plotting | `pip install matplotlib` |
| Seaborn | Statistical plotting | `pip install seaborn` |
| Scikit-Learn | Machine learning | `pip install scikit-learn` |
| TensorFlow | Deep learning | `pip install tensorflow` |
| PyTorch | Deep learning | `pip install torch` |
| Requests | HTTP requests | `pip install requests` |
| Jupyter | Interactive notebooks | `pip install jupyter` |
| Anaconda | Everything bundled | Download from anaconda.com |

### `requirements.txt` — Dependency Management

**File: `requirements.txt`**
```
numpy==1.24.0
pandas>=2.0,<3.0
scikit-learn==1.3.0
matplotlib>=3.5.0
```

**Generate from current environment:**
```bash
pip freeze > requirements.txt
```

**Install from file:**
```bash
pip install -r requirements.txt
```

> ⚠️ **Special Case — Version pinning**: Always pin versions in `requirements.txt` for reproducibility. `numpy>=1.20` works but different versions may behave differently. `numpy==1.24.0` is safer for production.

> **Data Science relevance**: Every ML project needs a `requirements.txt`. Without it, collaborators can't reproduce your environment, leading to "works on my machine" bugs.

In [ ]:
# --- pip Commands (informational) ---
# These show how to use pip, but won't execute in notebook

pip_commands = """
# List installed packages
pip list

# Install a package
pip install requests

# Install specific version
pip install numpy==1.24.0

# Install multiple packages
pip install numpy pandas matplotlib

# Install from requirements file
pip install -r requirements.txt

# Show package info
pip show pandas

# Save current environment
pip freeze > requirements.txt

# Upgrade package
pip install --upgrade scikit-learn

# Uninstall package
pip uninstall requests
"""

print(pip_commands)

# Create example requirements.txt
requirements = """# Data Science Stack
numpy==1.24.0
pandas>=2.0,<3.0
scikit-learn==1.3.0
matplotlib>=3.5.0
seaborn>=0.12.0
jupyter>=1.0.0
requests>=2.28.0
"""

with open('requirements.txt', 'w') as f:
    f.write(requirements)

print("\n✅ Created requirements.txt")
print("\nContents:")
print(requirements)

In [ ]:
# --- Check Installed Packages (within notebook) ---
import subprocess
import sys

# Get pip version
result = subprocess.run([sys.executable, '-m', 'pip', '--version'], 
                       capture_output=True, text=True)
print("pip version:")
print(result.stdout)

# List some key packages
packages_to_check = ['numpy', 'pandas', 'matplotlib', 'sklearn']
print("\nKey packages:")
for pkg in packages_to_check:
    result = subprocess.run([sys.executable, '-m', 'pip', 'show', pkg],
                           capture_output=True, text=True)
    if result.returncode == 0:
        lines = result.stdout.split('\n')
        name = lines[0].split(': ')[1]
        version = lines[1].split(': ')[1]
        print(f"  ✅ {name}: {version}")
    else:
        print(f"  ❌ {pkg}: not installed")

---
## 4. Virtual Environments — Isolated Python Workspaces

A **virtual environment** is an isolated Python installation for a specific project. Each environment has its own installed packages and Python version.

### Why Use Virtual Environments?

**Problem:** Global Python installation
```
Project A needs: pandas==1.3.0, tensorflow==2.8
Project B needs: pandas==2.0.0, tensorflow==2.13
→ Version conflict! Can only install one.
```

**Solution:** Virtual environments
```
venv_A/ (project A)
  ├── pandas==1.3.0
  └── tensorflow==2.8

venv_B/ (project B)
  ├── pandas==2.0.0
  └── tensorflow==2.13

Each project has its own isolated dependencies!
```

### Create and Activate Virtual Environment

#### On Mac/Linux:
```bash
# Create virtual environment
python3 -m venv my_project_env

# Activate it
source my_project_env/bin/activate

# Deactivate
deactivate
```

#### On Windows:
```bash
# Create virtual environment
python -m venv my_project_env

# Activate it
my_project_env\Scripts\activate

# Deactivate
deactivate
```

### Using Virtual Environments

| Step | Command |
|------|----------|
| 1. Create env | `python -m venv env_name` |
| 2. Activate | `source env_name/bin/activate` (Mac/Linux) or `env_name\Scripts\activate` (Windows) |
| 3. Install packages | `pip install pandas numpy` (only in this env!) |
| 4. Save dependencies | `pip freeze > requirements.txt` |
| 5. Deactivate | `deactivate` |

### Check if You're in a Virtual Environment

```bash
# Look at command prompt — should show (env_name) prefix:
(my_project_env) $ pip list

# Or check Python path:
which python   # Shows /path/to/env_name/bin/python
```

> ⚠️ **Special Case — Don't commit venv to Git**: Virtual environments are large and specific to each machine. Add `venv/` to `.gitignore` and share `requirements.txt` instead.

> ⚠️ **Special Case — Always activate before installing**: Forgetting to activate puts packages in the global Python (defeats the purpose!).

> **Data Science Best Practice**: EVERY project gets its own virtual environment. This is non-negotiable for professional work.

In [ ]:
# --- Virtual Environment Demo (informational) ---

demo_steps = """
# Workflow Example:

# 1. Create project directory
mkdir my_ml_project
cd my_ml_project

# 2. Create virtual environment
python -m venv venv

# 3. Activate it
source venv/bin/activate    # On Mac/Linux
# OR
venv\\Scripts\\activate     # On Windows

# 4. Upgrade pip (optional but recommended)
pip install --upgrade pip

# 5. Install dependencies
pip install numpy pandas scikit-learn jupyter

# 6. Save requirements
pip freeze > requirements.txt

# 7. Work on your project
python train_model.py
jupyter notebook

# 8. When done, deactivate
deactivate

# 9. Later, to continue work:
source venv/bin/activate
pip install -r requirements.txt  # Reinstall dependencies
"""

print(demo_steps)

# Show what goes into .gitignore
gitignore = """# Virtual Environment
venv/
env/
.venv/

# Python cache
__pycache__/
*.pyc

# IDE
.vscode/
.idea/

# OS
.DS_Store

# Notebooks
.ipynb_checkpoints/
"""

print("\n\n.gitignore file:")
print(gitignore)

### Conda — Alternative to venv

**Anaconda** provides an alternative environment manager called **conda**. It's more powerful but heavier than venv.

| Feature | venv | conda |
|---------|------|-------|
| Size | Small (minimal) | Large (pre-packaged) |
| Setup | Built-in, minimal | Requires Anaconda installation |
| Package selection | Huge (PyPI) | Large but curated |
| Speed | Fast | Slower (dependency resolution) |
| Use case | Standard Python | Scientific computing (pre-loaded NumPy, Pandas) |
| Recommendation | **Use this** | Use if you prefer bundled packages |

**Conda commands** (similar to venv):
```bash
# Create environment
conda create -n my_env python=3.11

# Activate
conda activate my_env

# Install packages
conda install numpy pandas scikit-learn

# Save environment
conda env export > environment.yml

# Deactivate
conda deactivate
```

> **Recommendation**: For beginners, use **venv** (built-in). For teams with non-Python dependencies, consider **conda**.

---
## 5. Best Practices & Project Structure

### Recommended Project Layout

```
my_data_science_project/
├── venv/                      # Virtual environment (don't commit!)
├── data/
│   ├── raw/                   # Original, unmodified data
│   ├── processed/             # Cleaned/transformed data
│   └── external/              # Data from APIs, databases
├── notebooks/
│   ├── 01_exploration.ipynb
│   ├── 02_preprocessing.ipynb
│   └── 03_modeling.ipynb
├── src/                       # Source code (your modules)
│   ├── __init__.py
│   ├── preprocessing.py       # Data cleaning functions
│   ├── features.py            # Feature engineering
│   ├── models.py              # Model training/evaluation
│   └── utils.py               # Helper functions
├── tests/
│   ├── test_preprocessing.py
│   └── test_models.py
├── .gitignore                 # Git ignore file
├── README.md                  # Project documentation
├── requirements.txt           # Dependencies
└── train.py                   # Main training script
```

### Key Files

**`README.md`** — Project overview
```markdown
# Customer Churn Prediction

Predicts which customers will churn using ML models.

## Setup

1. Create virtual environment: `python -m venv venv`
2. Activate: `source venv/bin/activate`
3. Install: `pip install -r requirements.txt`
4. Run: `python train.py`

## Results

Best model: Random Forest, 92% accuracy
```

**`.gitignore`** — Don't commit these
```
venv/
__pycache__/
*.pyc
.ipynb_checkpoints/
.vscode/
data/raw/
```

### Import Your Code in Notebooks

**Notebook: `notebooks/03_modeling.ipynb`**
```python
import sys
sys.path.insert(0, '../src')

from preprocessing import clean_data
from models import train_model, evaluate

# Use your modules
clean_df = clean_data(raw_df)
model = train_model(clean_df)
```

> **Data Science Best Practices**:
> 1. One virtual environment per project
> 2. Keep notebooks for exploration; move logic to `.py` files
> 3. Every project gets a README
> 4. Version control everything except data and venv
> 5. Document your modules with docstrings

In [ ]:
# --- Create Example Project Structure ---
from pathlib import Path

# Create directories
project = Path('example_project')
project.mkdir(exist_ok=True)

for dir_name in ['data/raw', 'data/processed', 'notebooks', 'src', 'tests']:
    (project / dir_name).mkdir(parents=True, exist_ok=True)

# Create README.md
readme = '''# Example Data Science Project

This project demonstrates best practices for organizing a data science project.

## Setup

```bash
python -m venv venv
source venv/bin/activate   # On Mac/Linux
pip install -r requirements.txt
```

## Project Structure

- `data/` — Raw and processed data
- `notebooks/` — Jupyter notebooks for exploration
- `src/` — Reusable Python modules
- `tests/` — Unit tests

## Running Analysis

```bash
python train.py
jupyter notebook
```
'''

with open(project / 'README.md', 'w') as f:
    f.write(readme)

# Create .gitignore
gitignore = '''venv/
env/
__pycache__/
*.pyc
.ipynb_checkpoints/
.DS_Store
data/raw/
.vscode/
.idea/
'''

with open(project / '.gitignore', 'w') as f:
    f.write(gitignore)

# Create requirements.txt
reqs = '''numpy>=1.20.0
pandas>=1.3.0
scikit-learn>=1.0.0
matplotlib>=3.4.0
jupyter>=1.0.0
'''

with open(project / 'requirements.txt', 'w') as f:
    f.write(reqs)

# Create __init__.py in src
with open(project / 'src' / '__init__.py', 'w') as f:
    f.write('"""Data science utilities."""\n')

# List structure
print("✅ Created project structure:")
for item in sorted(project.rglob('*')):
    level = len(item.relative_to(project).parts)
    indent = '  ' * level
    if item.is_dir():
        print(f"{indent}📁 {item.name}/")
    else:
        print(f"{indent}📄 {item.name}")

In [ ]:
# --- Example: Using Your Own Module in Real Code ---

# Create a realistic preprocessing module
src_code = '''"""Data preprocessing utilities for ML pipeline.

This module provides functions for:
- Loading data
- Cleaning missing values
- Scaling features

Example:
    from preprocessing import load_data, clean_data
    
    df = load_data('data.csv')
    clean_df = clean_data(df)
"""

import statistics

def load_data(filename):
    """Load CSV data file."""
    import csv
    with open(filename, 'r') as f:
        reader = csv.DictReader(f)
        return list(reader)

def fill_missing(data, column, strategy='mean'):
    """Fill missing values in a column.
    
    Args:
        data: List of dicts (records)
        column: Column name to fill
        strategy: 'mean' or 'median' or 'drop'
    
    Returns:
        List of dicts with filled values
    """
    if strategy == 'drop':
        return [r for r in data if r[column] != '']
    
    # Calculate mean/median
    values = [float(r[column]) for r in data if r[column] != '']
    if not values:
        return data
    
    if strategy == 'mean':
        fill_value = statistics.mean(values)
    else:  # median
        fill_value = statistics.median(values)
    
    # Fill missing
    result = []
    for r in data:
        if r[column] == '':
            r[column] = str(fill_value)
        result.append(r)
    return result

def scale_numeric(value, min_val, max_val):
    """Scale a value to 0-1 range."""
    return (float(value) - min_val) / (max_val - min_val)

if __name__ == '__main__':
    # Test the module
    test_data = [
        {'name': 'Alice', 'age': '30', 'score': '92'},
        {'name': 'Bob', 'age': '', 'score': '85'},
        {'name': 'Charlie', 'age': '35', 'score': ''}
    ]
    print(f"Original: {test_data}")
    clean = fill_missing(test_data, 'age', 'mean')
    print(f"Cleaned: {clean}")
'''

with open('preprocessing.py', 'w') as f:
    f.write(src_code)

print("✅ Created realistic preprocessing module")

# Use it
from preprocessing import fill_missing

test_data = [
    {'name': 'Alice', 'age': '30'},
    {'name': 'Bob', 'age': ''},
    {'name': 'Charlie', 'age': '35'}
]

print(f"\nBefore: {test_data}")
cleaned = fill_missing(test_data, 'age', 'mean')
print(f"After:  {cleaned}")

---
## 6. Practice Exercises

Apply what you've learned about modules, packages, and environments.

### Exercise 1: Create a Statistics Module
Create a module `stats_utils.py` with functions:
- `mean(data)` — Calculate average
- `median(data)` — Calculate median
- `std_dev(data)` — Calculate standard deviation
- `z_score(value, data)` — Calculate z-score

Include docstrings and test with `if __name__ == '__main__':`

In [ ]:
# Exercise 1: Statistics Module

# Your code here:

### Exercise 2: Create a Data Science Package
Build a package `ml_utils` with modules:
1. `preprocessing.py` — Functions for data cleaning
2. `validation.py` — Functions for input validation
3. `__init__.py` — Export main functions

Test importing from the package.

In [ ]:
# Exercise 2: Data Science Package

# Your code here:

### Exercise 3: Create requirements.txt
Create a `requirements.txt` file for a typical data science project with:
- NumPy (pinned version)
- Pandas (version range)
- Scikit-Learn, Matplotlib, Jupyter

Document the purpose of each package.

In [ ]:
# Exercise 3: Create requirements.txt

# Your code here:

### Exercise 4: Organize a Project
Create a mini project structure with:
- `data/` directory (raw + processed)
- `src/` package with a preprocessing module
- `notebooks/` directory (empty)
- `README.md`, `requirements.txt`, `.gitignore`

Document the structure in README.

In [ ]:
# Exercise 4: Organize a Project

# Your code here:

### Exercise 5: Virtual Environment Walkthrough
Document the steps to:
1. Create a virtual environment named `ds_project`
2. Activate it
3. Install packages from `requirements.txt`
4. Save the current environment state
5. Deactivate

Write commands for both Windows and Mac/Linux.

In [ ]:
# Exercise 5: Virtual Environment Walkthrough

# Your code here:

---
## 📝 Session 5 Summary

### What You Learned

| Topic | Key Takeaway |
|-------|---------------|
| **Modules** | Write `.py` files to reuse code; use `import` to load them |
| **`__name__`** | Use `if __name__ == '__main__':` to separate module code from test code |
| **Packages** | Organize modules in directories with `__init__.py` |
| **`pip`** | Install packages: `pip install package`, save requirements: `pip freeze > requirements.txt` |
| **Virtual Environments** | Isolate projects: `python -m venv venv`, `source venv/bin/activate` |
| **Project Structure** | Standard layout: `data/`, `src/`, `notebooks/`, `tests/`, README, requirements.txt |

### Key Gotchas to Remember
- `from module import *` is dangerous — always specify what you need
- Always use `if __name__ == '__main__':` to separate test code
- Circular imports cause errors — redesign module dependencies
- Relative imports (with `.`) only work inside packages, not when running directly
- Pin versions in `requirements.txt` for reproducibility
- Don't commit `venv/` to Git — add to `.gitignore`
- Always activate virtual env BEFORE installing packages

### Next Session
**Session 6**: Object-Oriented Programming (OOP) — Classes, Objects, Inheritance — building larger applications professionally.